In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Set display options for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the datasets
data_dir = Path("data")

# Load reference dataset
ref_data_path = data_dir / "reference_data.csv"
new_data_path = data_dir / "new_data_with_drift.csv"

ref_df = pd.read_csv(ref_data_path)
new_df = pd.read_csv(new_data_path)

print(f"Reference dataset loaded: {ref_df.shape[0]} rows, {ref_df.shape[1]} columns")
print(f"New dataset loaded: {new_df.shape[0]} rows, {new_df.shape[1]} columns")


Reference dataset loaded: 80 rows, 8 columns
New dataset loaded: 80 rows, 9 columns


In [3]:
# Display all columns for both datasets
print("=" * 80)
print("COLUMNS IN REFERENCE DATASET")
print("=" * 80)
print(f"Columns ({len(ref_df.columns)}):")
for i, col in enumerate(ref_df.columns, 1):
    print(f"  {i}. {col}")

print("\n" + "=" * 80)
print("COLUMNS IN NEW DATASET")
print("=" * 80)
print(f"Columns ({len(new_df.columns)}):")
for i, col in enumerate(new_df.columns, 1):
    print(f"  {i}. {col}")

print("\n" + "=" * 80)
print("COLUMN DIFFERENCES")
print("=" * 80)
ref_cols = set(ref_df.columns)
new_cols = set(new_df.columns)
only_in_ref = ref_cols - new_cols
only_in_new = new_cols - ref_cols
common_cols = ref_cols & new_cols

if only_in_ref:
    print(f"Columns only in Reference: {only_in_ref}")
if only_in_new:
    print(f"Columns only in New: {only_in_new}")
if common_cols:
    print(f"Common columns: {len(common_cols)} columns")


COLUMNS IN REFERENCE DATASET
Columns (8):
  1. order_id
  2. customer_id
  3. order_date
  4. region
  5. product_category
  6. order_amount
  7. payment_method
  8. is_priority

COLUMNS IN NEW DATASET
Columns (9):
  1. order_id
  2. customer_id
  3. order_date
  4. region
  5. product_category
  6. order_amount
  7. payment_method
  8. is_priority
  9. discount_code

COLUMN DIFFERENCES
Columns only in New: {'discount_code'}
Common columns: 8 columns


In [4]:
# Data types for both datasets
print("=" * 80)
print("DATA TYPES - REFERENCE DATASET")
print("=" * 80)
print(ref_df.dtypes)
print(f"\nTotal columns: {len(ref_df.columns)}")
print(f"Memory usage: {ref_df.memory_usage(deep=True).sum() / 1024:.2f} KB")

print("\n" + "=" * 80)
print("DATA TYPES - NEW DATASET")
print("=" * 80)
print(new_df.dtypes)
print(f"\nTotal columns: {len(new_df.columns)}")
print(f"Memory usage: {new_df.memory_usage(deep=True).sum() / 1024:.2f} KB")


DATA TYPES - REFERENCE DATASET
order_id             object
customer_id          object
order_date           object
region               object
product_category     object
order_amount        float64
payment_method       object
is_priority            bool
dtype: object

Total columns: 8
Memory usage: 27.96 KB

DATA TYPES - NEW DATASET
order_id            object
customer_id         object
order_date          object
region              object
product_category    object
order_amount        object
payment_method      object
is_priority           bool
discount_code       object
dtype: object

Total columns: 9
Memory usage: 36.34 KB


In [5]:
# Null values analysis
print("=" * 80)
print("NULL VALUES - REFERENCE DATASET")
print("=" * 80)
null_ref = ref_df.isnull().sum()
null_pct_ref = (null_ref / len(ref_df)) * 100
null_summary_ref = pd.DataFrame({
    'Column': null_ref.index,
    'Null Count': null_ref.values,
    'Null Percentage': null_pct_ref.values
})
null_summary_ref = null_summary_ref[null_summary_ref['Null Count'] > 0].sort_values('Null Count', ascending=False)
if len(null_summary_ref) > 0:
    print(null_summary_ref.to_string(index=False))
else:
    print("No null values found!")
print(f"\nTotal null values: {null_ref.sum()}")
print(f"Rows with at least one null: {(ref_df.isnull().any(axis=1)).sum()}")

print("\n" + "=" * 80)
print("NULL VALUES - NEW DATASET")
print("=" * 80)
null_new = new_df.isnull().sum()
null_pct_new = (null_new / len(new_df)) * 100
null_summary_new = pd.DataFrame({
    'Column': null_new.index,
    'Null Count': null_new.values,
    'Null Percentage': null_pct_new.values
})
null_summary_new = null_summary_new[null_summary_new['Null Count'] > 0].sort_values('Null Count', ascending=False)
if len(null_summary_new) > 0:
    print(null_summary_new.to_string(index=False))
else:
    print("No null values found!")
print(f"\nTotal null values: {null_new.sum()}")
print(f"Rows with at least one null: {(new_df.isnull().any(axis=1)).sum()}")


NULL VALUES - REFERENCE DATASET
No null values found!

Total null values: 0
Rows with at least one null: 0

NULL VALUES - NEW DATASET
       Column  Null Count  Null Percentage
discount_code          16             20.0

Total null values: 16
Rows with at least one null: 16


In [6]:
# Descriptive statistics - Reference Dataset
print("=" * 80)
print("DESCRIPTIVE STATISTICS - REFERENCE DATASET")
print("=" * 80)
print("\nShape:", ref_df.shape)
print("\nFirst few rows:")
print(ref_df.head())
print("\n" + "-" * 80)
print("Statistical Summary:")
print(ref_df.describe(include='all'))
print("\n" + "-" * 80)
print("Basic Info:")
print(f"Total rows: {len(ref_df)}")
print(f"Total columns: {len(ref_df.columns)}")
print(f"Duplicate rows: {ref_df.duplicated().sum()}")


DESCRIPTIVE STATISTICS - REFERENCE DATASET

Shape: (80, 8)

First few rows:
        order_id customer_id  order_date region product_category  \
0  ORD-2023-0001   CUST-1001  2023-01-03  North      Electronics   
1  ORD-2023-0002   CUST-1002  2023-01-04  South             Home   
2  ORD-2023-0003   CUST-1003  2023-01-05   East            Books   
3  ORD-2023-0004   CUST-1004  2023-01-06   West      Electronics   
4  ORD-2023-0005   CUST-1005  2023-01-07  North         Clothing   

   order_amount payment_method  is_priority  
0        249.99    Credit Card        False  
1         89.50     Debit Card        False  
2         19.99         PayPal        False  
3       1299.00    Credit Card         True  
4         59.99     Debit Card        False  

--------------------------------------------------------------------------------
Statistical Summary:
             order_id customer_id  order_date region product_category  \
count              80          80          80     80           

In [7]:
# Descriptive statistics - New Dataset
print("=" * 80)
print("DESCRIPTIVE STATISTICS - NEW DATASET")
print("=" * 80)
print("\nShape:", new_df.shape)
print("\nFirst few rows:")
print(new_df.head())
print("\n" + "-" * 80)
print("Statistical Summary:")
print(new_df.describe(include='all'))
print("\n" + "-" * 80)
print("Basic Info:")
print(f"Total rows: {len(new_df)}")
print(f"Total columns: {len(new_df.columns)}")
print(f"Duplicate rows: {new_df.duplicated().sum()}")


DESCRIPTIVE STATISTICS - NEW DATASET

Shape: (80, 9)

First few rows:
          order_id customer_id  order_date         region product_category  \
0  ORDER-2024-A001   CUST-A201  2024-02-03  North America      Electronics   
1  ORDER-2024-A002   CUST-A202  2024-02-04           EMEA       Industrial   
2  ORDER-2024-A003   CUST-A203  2024-02-05           APAC         Software   
3  ORDER-2024-A004   CUST-A204  2024-02-06  North America      Electronics   
4  ORDER-2024-A005   CUST-A205  2024-02-07           EMEA       Industrial   

  order_amount payment_method  is_priority discount_code  
0      5499.99         Crypto         True     NEWYEAR50  
1     12999.00  Wire Transfer         True         B2B10  
2       799.00    Credit Card         True      SPRING25  
3      2999.99           Null         True     NEWYEAR50  
4     18999.00  Wire Transfer         True           NaN  

--------------------------------------------------------------------------------
Statistical Summary:
    

In [8]:
# Quick comparison summary
print("=" * 80)
print("DATASET COMPARISON SUMMARY")
print("=" * 80)

comparison = pd.DataFrame({
    'Metric': ['Total Rows', 'Total Columns', 'Null Values', 'Duplicate Rows'],
    'Reference Dataset': [
        len(ref_df),
        len(ref_df.columns),
        ref_df.isnull().sum().sum(),
        ref_df.duplicated().sum()
    ],
    'New Dataset': [
        len(new_df),
        len(new_df.columns),
        new_df.isnull().sum().sum(),
        new_df.duplicated().sum()
    ],
    'Difference': [
        len(new_df) - len(ref_df),
        len(new_df.columns) - len(ref_df.columns),
        new_df.isnull().sum().sum() - ref_df.isnull().sum().sum(),
        new_df.duplicated().sum() - ref_df.duplicated().sum()
    ]
})

print(comparison.to_string(index=False))


DATASET COMPARISON SUMMARY
        Metric  Reference Dataset  New Dataset  Difference
    Total Rows                 80           80           0
 Total Columns                  8            9           1
   Null Values                  0           16          16
Duplicate Rows                  0            0           0
